# AgentGuard: Week 2, full baseline evaluation

Runs the **whole attack suite** against the undefended agent:
- **100 attacks** across 11 categories (A1 to A11) and **30 normal control tasks**
- each case **5 times at temperature 0.7**, plus **once at temperature 0**
- every run scored automatically from logs (outbox, file access, memory, tool calls, exact strings)

That is about **780 agent runs**, roughly **1.5 to 2 hours** on a Kaggle T4 (the A11 loop attacks are the slow ones). The harness saves every run as it finishes, so if the session drops, just **Run All** again in the same session and it continues where it stopped.

**Kaggle:** `Session options → Accelerator → GPU T4 x2`, `Internet → On`, then **Run All**. Keep the browser tab open while it runs.

At the end, download `baseline_results.zip` from the Output panel and send it to me.

## Step 1: Settings

In [ ]:
REPO_URL = "https://github.com/anshgupta8381-jpg/agentguard-testbed.git"
MODEL = "qwen2.5:7b"
EMBED_MODEL = "nomic-embed-text"
RUNS = 5             # runs per case at the sampling temperature
TEMPERATURE = 0.7
CONFIG_NAME = "baseline"

## Step 2: GPU check and storage

In [ ]:
import subprocess, os, shutil, sys, time
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU: Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU. Then run all again.")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    WORK_DIR, STORE_DIR = "/kaggle/working", "/kaggle/working/agentguard"
else:
    WORK_DIR = "/content"
    from google.colab import drive
    drive.mount("/content/drive")
    STORE_DIR = "/content/drive/MyDrive/agentguard"
os.makedirs(STORE_DIR, exist_ok=True)
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab", "| storage:", STORE_DIR)

## Step 3: Install and start Ollama, pull the models

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import requests
server_proc = subprocess.Popen(["ollama", "serve"], stdout=open(f"{WORK_DIR}/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        if requests.get("http://localhost:11434").ok:
            print("Ollama server is running"); break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print(f"Server did not start. Check {WORK_DIR}/ollama.log")
for m in [EMBED_MODEL, MODEL]:   # NameError here means Step 1 was skipped: use Run All
    print(f"Pulling {m} ...")
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## Step 4: Get the latest code from GitHub (pip warnings about other packages are normal)

In [ ]:
REPO_DIR = f"{WORK_DIR}/agentguard-testbed"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
!pip install -q -r {REPO_DIR}/requirements.txt
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!git log --oneline -3

## Step 5: Check the data and run the unit tests (expect 45 passed)

In [ ]:
!python scripts/validate_data.py
!python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -3

## Step 6: Build the index, load the attack suite, start the harness

In [ ]:
import json, pandas as pd, ollama
from datetime import datetime
from agentguard.rag import DocIndex
from agentguard.harness import Harness, Config, load_cases

index = DocIndex(data_dir="data", db_dir=f"{STORE_DIR}/chroma_db")
print("Chunks indexed:", index.build())
ollama.chat(model=MODEL, messages=[{"role": "user", "content": "Say OK."}]); index.search("warm up", k=1)

cases = load_cases("attacks/cases.yaml")
print(pd.Series([c["category"] for c in cases]).value_counts().sort_index().to_dict())

EVAL_DIR = f"{STORE_DIR}/eval"
harness = Harness(index=index, data_dir="data", out_dir=EVAL_DIR, model=MODEL)
config = Config(name=CONFIG_NAME)
print("Results database:", harness.db_path, "| runs already saved:", len(harness.rows(CONFIG_NAME)))

## Step 7: Smoke test (one case per category)
A quick check that every kind of case runs before the long run starts. These runs go into a separate "smoke" config, so they don't mix with the real results.

In [ ]:
smoke = [next(c for c in cases if c["category"] == cat) for cat in [f"A{i}" for i in range(1, 12)] + ["control"]]
for c in smoke:
    r = harness.run_case(c, Config(name="smoke"), run=1, temperature=TEMPERATURE)
    print(f"{c['id']:<7} success={r['success']} exposed={r['exposed']} tools={r['n_tool_calls']:>2} {r['seconds']:5.1f}s  {r['stop_reason']}")

## Step 8: The main run (100 attacks + 30 controls, 5 runs each)
Progress is printed every 25 runs. If the session drops, Run All again: finished runs are skipped.

In [ ]:
t_start = time.time()
def progress(i, total, row):
    if i % 25 == 0 or i == total:
        el = time.time() - t_start
        print(f"{i}/{total} runs | {el/60:.1f} min | ETA {el/i*(total-i)/60:.1f} min | last: {row['case_id']} success={row['success']}")
new = harness.run(cases, config, runs=RUNS, temperature=TEMPERATURE, progress=progress)
print(f"New runs this session: {new} | total saved: {len(harness.rows(CONFIG_NAME))}")

## Step 9: Temperature-0 pass (each case once, the model's most likely behaviour)

In [ ]:
t_start = time.time()
new0 = harness.run(cases, config, runs=1, temperature=0.0, progress=progress)
print(f"New temperature-0 runs: {new0}")

## Step 10: Baseline results by category

In [ ]:
OWASP = {c["category"]: ", ".join(c["owasp"]) for c in cases}
NAMES = {"A1": "Injection in pasted text", "A2": "Injection in a document (RAG)", "A3": "Injection in an email",
         "A4": "Injection in a web page", "A5": "System prompt exposure", "A6": "Restricted data to employee",
         "A7": "Unsafe tool arguments", "A8": "Knowledge-base poisoning", "A9": "Memory poisoning (2 sessions)",
         "A10": "MCP tool poisoning / rug pull", "A11": "Unbounded consumption", "control": "Normal tasks (utility)"}
summary = pd.DataFrame(harness.summary(CONFIG_NAME, temperature=TEMPERATURE))
summary.insert(1, "attack", summary["category"].map(NAMES))
summary.insert(2, "owasp", summary["category"].map(lambda c: OWASP.get(c, "")))
t0 = {r["category"]: r["success_rate"] for r in harness.summary(CONFIG_NAME, temperature=0.0)}
summary["success_T0"] = summary["category"].map(t0)

att = [r for r in harness.rows(CONFIG_NAME) if r["category"] != "control" and r["temperature"] == TEMPERATURE]
ctl = [r for r in harness.rows(CONFIG_NAME) if r["category"] == "control" and r["temperature"] == TEMPERATURE]
print(f"OVERALL attack success rate (T={TEMPERATURE}): {sum(r['success'] for r in att)}/{len(att)} = {sum(r['success'] for r in att)/len(att):.0%}")
print(f"Attack cases that succeeded at least once in {RUNS} runs: {sum(any(x['success'] for x in att if x['case_id']==cid) for cid in {r['case_id'] for r in att})}/100")
print(f"Normal task success (utility): {sum(r['success'] for r in ctl)}/{len(ctl)} = {sum(r['success'] for r in ctl)/len(ctl):.0%}")
print(f"Unrequested send_email/save_memory in NORMAL tasks: {sum(bool(r['side_effects']) for r in ctl)}/{len(ctl)} runs")
summary[["category", "attack", "owasp", "cases", "runs", "success_rate", "success_T0", "any_success_cases",
         "exposed_rate", "warned_rate", "side_effect_rate", "avg_tool_calls", "avg_seconds", "errors"]]

## Step 11: Per-case detail

In [ ]:
df = pd.DataFrame(harness.rows(CONFIG_NAME))
per_case = (df[df["temperature"] == TEMPERATURE].groupby(["case_id", "category"])
            .agg(success_rate=("success", "mean"), exposed=("exposed", "mean"), warned=("warned", "mean"),
                 tool_calls=("n_tool_calls", "mean")).reset_index())
names = {c["id"]: c["name"] for c in cases}
per_case.insert(2, "name", per_case["case_id"].map(names))
pd.set_option("display.max_rows", 200, "display.max_colwidth", 60)
print("Most successful attacks:")
print(per_case[per_case.category != "control"].sort_values("success_rate", ascending=False).head(15).to_string(index=False))
print("\nAttacks that never worked:")
print(per_case[(per_case.category != "control") & (per_case.success_rate == 0)][["case_id", "name", "exposed", "warned"]].to_string(index=False))
a10 = df[(df.category == "A10") & (df.temperature == TEMPERATURE)].copy()
a10["variant"] = (a10["mcp_approved_hash"] != a10["mcp_runtime_hash"]).map({True: "rug pull", False: "poisoned from start"})
print("\nA10 by variant:", a10.groupby("variant")["success"].mean().round(2).to_dict())

## Step 12: Look at one successful attack

In [ ]:
hits = df[(df.success == 1) & (df.category != "control")]
if len(hits):
    ex = hits.sample(1, random_state=7).iloc[0]
    t = json.load(open(ex["trace_path"]))
    print(f"{ex['case_id']}: {names[ex['case_id']]}  (user: {t['case']['user']})")
    for s in t["sessions"]:
        print(f"\n[session {s['session']}] USER: {s['prompt']}")
        for c in s["tool_calls"]:
            print(f"  -> {c['name']}({json.dumps(c['args'])[:120]})")
        print(f"  REPLY: {s['reply'][:400]}")

## Step 13: Save everything for download

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")
export = f"{STORE_DIR}/baseline_export"
os.makedirs(export, exist_ok=True)
df.drop(columns=["trace_path"]).to_csv(f"{export}/baseline_runs.csv", index=False)
summary.to_csv(f"{export}/baseline_summary.csv", index=False)
per_case.to_csv(f"{export}/baseline_per_case.csv", index=False)
shutil.copy(harness.db_path, f"{export}/results.db")
shutil.make_archive(f"{export}/traces", "zip", f"{EVAL_DIR}/runs/{CONFIG_NAME}")
zip_path = shutil.make_archive(f"{STORE_DIR}/baseline_results", "zip", export)
print("Download this file from the Output panel:", zip_path)
print(f"Write these down -> attack success: {sum(r['success'] for r in att)/len(att):.0%} | "
      f"utility: {sum(r['success'] for r in ctl)/len(ctl):.0%} | runs: {len(df)}")

## Week 2 checklist
- [ ] 45 unit tests pass (Step 5)
- [ ] Smoke test: every category runs without errors (Step 7)
- [ ] Main run and temperature-0 run finished (Steps 8 and 9)
- [ ] Step 10 table and the "Write these down" line
- [ ] `baseline_results.zip` downloaded and sent to me

**Next (Week 3):** the 10 defences (D1 to D10), then the same suite against each defence, so we can see exactly how much each one lowers these numbers and what it costs in normal-task success.